# SelectOmics · Real-World Benchmark, TCGA Lower-Grade Glioma (LGG)

**Dataset:** TCGA-LGG multi-omics, 247 samples × 34,069 features across four omic
layers (mRNA · DNA-methylation · miRNA · CNV).
**Task:** 3-class molecular-subtype classification (labels 0 / 1 / 2, 76 / 125 / 46).
**Goal:** evaluate SelectOmics (with the benchmark-tuned `omics` preset) against
standard feature selectors on **real** omics data, where there is **no ground-truth
informative feature set**.

Because there is no ground truth, this benchmark replaces the synthetic suite's
F1-vs-truth with the axes that actually matter on real data:

| Axis | Metric |
|------|--------|
| Downstream prediction | macro one-vs-rest AUC · macro-F1 · balanced accuracy |
| **Stability** | Kuncheva Index across CV folds (the headline differentiator) |
| Parsimony | n_selected · reduction ratio · residual redundancy |
| Biological soft-truth | recovery of known glioma driver genes |

Each omic **layer is benchmarked separately**. Feature selection is **refit inside
every CV fold** (no leakage); median imputation is fit on the train split only.
Features are *not* re-standardised, the layers are pre-normalised upstream, and
forcing unit variance would neutralise the variance-based steps (SelectOmics
Step 1, the VarCorr baseline).

## 1 · Setup

In [1]:
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    display = print

# benchmark_lgg.py lives in this directory; ensure it is importable.
_HERE = Path(os.path.abspath('')).resolve()
for _p in (str(_HERE), str(_HERE.parent)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import benchmark_lgg as bl

print('benchmark_lgg loaded')
print(f'  SHAP available        : {bl._SHAP_AVAILABLE}')
print(f'  XGBoost available     : {bl._XGB_AVAILABLE}')
print(f'  SelectOmics available : {bl._SO_AVAILABLE}')

benchmark_lgg loaded
  SHAP available        : True
  XGBoost available     : True
  SelectOmics available : True


## 2 · Pre-registration (locked benchmark specification)

Declared **before** inspecting any result. Changing metrics, the CV protocol,
or the layer list after seeing results constitutes p-hacking.

In [2]:
LGG_BENCHMARK_SPEC = {
    'schema_version': '1.0',
    'dataset':        'TCGA-LGG multi-omics (247 samples, 3-class subtype)',
    'layers':         bl.LAYERS,                       # benchmarked separately
    'primary_metrics': {
        'downstream_performance': 'auc_ovr',           # macro one-vs-rest AUC
        'stability':              'kuncheva',          # across CV folds
        'parsimony':              'n_selected',
    },
    'secondary_metrics': ['f1_macro', 'balanced_acc', 'reduction_ratio', 'redundancy'],
    'cv_protocol':       'RepeatedStratifiedKFold; selection refit per fold (no leakage)',
    'preprocessing':     'median impute (train-fit); NO re-standardisation '
                         '(layers pre-normalised; preserves variance-based steps)',
    'statistical_test':  'friedman_then_wilcoxon_signed_rank_vs_SelectOmics',
    'correction':        'holm_bonferroni',
    'effect_size':       'cliffs_delta',
    'alpha':             0.05,
    'note':              'Real data has no ground-truth feature set; F1-vs-truth is '
                         'replaced by downstream prediction + stability + parsimony + biology.',
}
print(json.dumps(LGG_BENCHMARK_SPEC, indent=2, default=str))

{
  "schema_version": "1.0",
  "dataset": "TCGA-LGG multi-omics (247 samples, 3-class subtype)",
  "layers": [
    "mRNA",
    "Methy",
    "miRNA",
    "CNV"
  ],
  "primary_metrics": {
    "downstream_performance": "auc_ovr",
    "stability": "kuncheva",
    "parsimony": "n_selected"
  },
  "secondary_metrics": [
    "f1_macro",
    "balanced_acc",
    "reduction_ratio",
    "redundancy"
  ],
  "cv_protocol": "RepeatedStratifiedKFold; selection refit per fold (no leakage)",
  "preprocessing": "median impute (train-fit); NO re-standardisation (layers pre-normalised; preserves variance-based steps)",
  "statistical_test": "friedman_then_wilcoxon_signed_rank_vs_SelectOmics",
  "correction": "holm_bonferroni",
  "effect_size": "cliffs_delta",
  "alpha": 0.05,
  "note": "Real data has no ground-truth feature set; F1-vs-truth is replaced by downstream prediction + stability + parsimony + biology."
}


## 3 · Configuration

**Pick layers and depth independently.** `LAYERS_TO_RUN` chooses *which* omic
layers to benchmark; `FAST_MODE` controls *only* fold count + SelectOmics depth.
Results **accumulate in one checkpoint CSV**, so you can run `['miRNA']` now and
`['mRNA']` later without redoing anything, and every report cell below reflects
**all** layers present in the CSV, not just the one you last ran.

| `FAST_MODE` | Folds | SelectOmics Step 3 | Use |
|-------------|-------|--------------------|-----|
| `True`  | 3×1 = 3  | off | quick look / wiring check |
| `False` | 5×2 = 10 | on  | real result (the ~11k-feature layers take hours **each**) |

The runner is **checkpoint-safe**: each (layer, method, fold) row is written
immediately, so an interrupted run resumes from the saved CSV. Set `RERUN=False`
to skip straight to analysis of an existing `results_lgg/` CSV.

> Two distinct experiments:
> - **Per-layer** (`bl.LAYERS`, or one at a time), which omic layer carries the
>   subtype signal; most interpretable; runtime is per-layer.
> - **Integrated** (`['merged']`), all 34k features in one model; tests whether
>   combining layers beats the best single layer. This is the **heaviest** run
>   (SelectOmics with Step 3 on 34k features is very slow), start with
>   `FAST_MODE=True` to gauge timing, and keep it in its own `OUTPUT_DIR`.

In [3]:
# ── EDIT ──────────────────────────────────────────────────────────────────
# What to run (entries accumulate into ONE CSV so you can compare side by side):
#   'mRNA' / 'Methy' / 'miRNA' / 'CNV' , a single omic layer
#   'merged'                           , ALL layers together (one 34k model)
#   bl.LAYERS                          , the four layers, each separately
# Per-layer and 'merged' share the same folds (same labels + seed), so they are
# directly comparable in one file.
LAYERS_TO_RUN = bl.LAYERS + ['merged']    # every layer + the integrated run
FAST_MODE     = False            # True = 3 folds (quick, but stats need >=4);
                                 # False = 10 folds + SO Step 3 ON (valid stats, slow)
RERUN         = True             # False = just load & analyse the existing CSV
OUTPUT_DIR    = Path('results_lgg')
# ⚠ If you CHANGE FAST_MODE, delete the existing CSV first, fold indices are
#   not comparable across CV settings and must never be mixed in one file.
# ──────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(exist_ok=True)
N_SPLITS, N_REPEATS, SO_FAST = (3, 1, True) if FAST_MODE else (5, 2, False)
# Fold count is baked into the filename, so 3-fold and 10-fold runs land in
# separate files and can never be accidentally mixed.
RAW_CSV = OUTPUT_DIR / f'benchmark_lgg_{N_SPLITS * N_REPEATS}fold.csv'

methods = bl.default_methods(so_fast=SO_FAST)
print(f'Layers to run : {LAYERS_TO_RUN}')
print(f'Folds         : {N_SPLITS}×{N_REPEATS} = {N_SPLITS * N_REPEATS}')
print(f'Methods       : {[m.name for m in methods]}')
print(f'SO Step 3     : {"OFF (fast)" if SO_FAST else "ON (full)"}')
print(f'Output        : {RAW_CSV}')

Layers to run : ['mRNA', 'Methy', 'miRNA', 'CNV', 'merged']
Folds         : 5×2 = 10
Methods       : ['VarCorr_Baseline', 'LASSO', 'ElasticNet', 'RFECV', 'RF_Importance', 'PermImportance', 'SHAP', 'SelectOmics']
SO Step 3     : ON (full)
Output        : results_lgg\benchmark_lgg_10fold.csv


## 4 · Data overview

In [4]:
df = bl.load_merged()
print(f'Merged matrix: {df.shape[0]} samples × {df.shape[1] - 1} features + Label')

rows = []
for layer in bl.LAYERS:
    cols = [c for c in df.columns if c.endswith(f'_{layer}')]
    X = df[cols].to_numpy(dtype=float)
    rows.append({'layer': layer, 'n_features': len(cols),
                 'missing_%': round(100 * np.isnan(X).mean(), 3)})
overview = pd.DataFrame(rows)
overview.loc[len(overview)] = {'layer': 'ALL', 'n_features': df.shape[1] - 1,
                               'missing_%': round(100 * np.isnan(
                                   df.drop(columns=[bl.TARGET_COL]).to_numpy(float)).mean(), 3)}
print('\nOmic layers:')
display(overview)

vc = df[bl.TARGET_COL].value_counts().sort_index()
print(f'\n3-class balance: {dict(vc)}  (imbalance {vc.max()/vc.min():.1f}:1)')

Merged matrix: 247 samples × 34069 features + Label

Omic layers:


,layer,n_features,missing_%
0,mRNA,11345,0.000
1,Methy,11191,0.170
2,miRNA,328,0.000
3,CNV,11205,0.000
4,ALL,34069,0.056



3-class balance: {0: np.int64(76), 1: np.int64(125), 2: np.int64(46)}  (imbalance 2.7:1)


## 5 · Run the benchmark

Iterates over the selected layers, benchmarking every method with repeated
stratified CV. Resumes automatically if `benchmark_lgg_raw.csv` already exists.

In [ ]:
if RERUN:
    # Guard: refuse to mix incompatible CV schemes in one CSV.
    _expected = N_SPLITS * N_REPEATS
    if RAW_CSV.exists():
        _seen = int(pd.read_csv(RAW_CSV)['fold'].max()) + 1
        if _seen != _expected:
            raise SystemExit(
                f"{RAW_CSV} already holds {_seen}-fold rows but FAST_MODE implies "
                f"{_expected} folds. Fold indices are not comparable across CV "
                f"schemes, delete the CSV (or change OUTPUT_DIR) before re-running.")
    for layer in LAYERS_TO_RUN:
        bl.run_layer(df, layer, methods, RAW_CSV,
                     n_splits=N_SPLITS, n_repeats=N_REPEATS, base_seed=0)
    results = pd.read_csv(RAW_CSV)
else:
    if not RAW_CSV.exists():
        raise FileNotFoundError(f'{RAW_CSV} not found, set RERUN=True.')
    results = pd.read_csv(RAW_CSV)

layers_present = sorted(results['layer'].unique())   # every layer in the CSV
print(f'\nRaw rows: {len(results)}  | layers present: {layers_present}')
summary = bl.summarise(results)
print('\n=== Per-layer summary (sorted by AUC within layer) ===')
display(summary)


-- Layer mRNA  [n=247 * p=11345 * 5x2 folds]
   parallelism: sequential x 24 thread(s) per estimator = 24 of 24 cores
   VarCorr_Baseline   fold=0  n_sel= 5521  AUC=0.9995  F1=0.9501  (0s)
   LASSO              fold=0  n_sel=  414  AUC=0.9986  F1=0.9501  (76s)
   ElasticNet         fold=0  n_sel=  726  AUC=0.9978  F1=0.9501  (82s)
   RFECV              fold=0  n_sel=  113  AUC=0.9945  F1=0.9501  (56s)
   RF_Importance      fold=0  n_sel=  926  AUC=0.9995  F1=0.9501  (0s)
   PermImportance     fold=0  n_sel=11345  AUC=0.9951  F1=0.9501  (132s)
   SHAP               fold=0  n_sel=  129  AUC=0.9951  F1=0.9501  (3s)
   SelectOmics        fold=0  n_sel=   16  AUC=0.9983  F1=0.9759  (568s)
   VarCorr_Baseline   fold=1  n_sel= 5510  AUC=0.9966  F1=0.9757  (0s)
   LASSO              fold=1  n_sel=  418  AUC=0.9952  F1=0.9461  (77s)
   ElasticNet         fold=1  n_sel=  735  AUC=0.9941  F1=0.9496  (81s)
   PermImportance     fold=7  n_sel=11345  AUC=1.0  F1=0.9696  (133s)
   SHAP              

## 6 · Downstream predictive performance

Macro one-vs-rest AUC of a fixed XGBoost classifier trained on each method's
selected features (held-out fold). On real omics, predictive parity is the bar;
the differentiation comes from stability and parsimony (Sections 7–8).

In [ ]:
for metric, label in [('auc_ovr', 'Macro-OVR AUC'),
                      ('f1_macro', 'Macro-F1'),
                      ('balanced_acc', 'Balanced accuracy')]:
    piv = summary.pivot(index='method', columns='layer', values=metric)
    print(f'\n{label}:')
    display(piv.style.background_gradient(cmap='viridis', axis=None).format('{:.3f}'))

In [ ]:
# Heatmap of macro-OVR AUC (method × layer)
piv = summary.pivot(index='method', columns='layer', values='auc_ovr')
fig, ax = plt.subplots(figsize=(max(5, 1.4 * piv.shape[1] + 3), 5))
sns.heatmap(piv, annot=True, fmt='.3f', cmap='viridis', linewidths=0.5, ax=ax)
ax.set_title('Downstream macro-OVR AUC  (method × layer)')
ax.set_xlabel(''); ax.set_ylabel('')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 7 · Stability, Kuncheva Index across folds

The headline differentiator for reproducible omics research: does the method
return the **same features** regardless of the data split? KI ∈ [−1, 1], near 1
is perfectly reproducible.

> Read KI together with `n_selected`: a method that keeps half the features
> (e.g. the variance baseline) is trivially stable. Stability *at low
> n_selected* is the meaningful combination.

In [ ]:
piv_ki = summary.pivot(index='method', columns='layer', values='kuncheva')
print('Kuncheva Stability Index (across folds):')
display(piv_ki.style.background_gradient(cmap='viridis', vmin=-1, vmax=1, axis=None).format('{:.3f}'))

# KI vs parsimony, one panel per layer.
#
# Wrapped into a grid rather than a single row. A notebook scales a figure down
# to the cell width, so five panels side by side arrive about two inches wide
# each and the method labels become unreadable. Two columns keeps every panel
# at a size that survives that scaling, and at print size in a paper.
import math

_n_layers = len(layers_present)
_ncols    = min(2, _n_layers)
_nrows    = math.ceil(_n_layers / _ncols)

fig, axes = plt.subplots(_nrows, _ncols,
                         figsize=(7.5 * _ncols, 5.6 * _nrows),
                         squeeze=False)

for ax, layer in zip(axes.ravel(), layers_present):
    sub = summary[summary['layer'] == layer]
    ax.scatter(sub['n_selected'], sub['kuncheva'],
               s=150, c=plt.cm.viridis(0.5), edgecolors='black',
               linewidths=1.0, zorder=3)
    for _, r in sub.iterrows():
        ax.annotate(r['method'], (r['n_selected'], r['kuncheva']),
                    textcoords='offset points', xytext=(8, 5), fontsize=11)

    ax.set_xscale('log')
    ax.set_xlabel('Features selected (log scale)', fontsize=13)
    ax.set_ylabel('Kuncheva Stability Index', fontsize=13)
    ax.set_title(f'{layer}: stability vs parsimony', fontsize=15)
    ax.tick_params(labelsize=11)
    # KI is bounded; fixing the axis makes panels comparable across layers.
    ax.set_ylim(-1.05, 1.05)
    ax.axhline(0.0, color='grey', lw=1.0, ls='--', alpha=0.6)
    ax.grid(alpha=0.3)
    # Headroom on the right so annotations do not run off the axes.
    ax.margins(x=0.22)

# Hide unused panels when the layer count does not fill the grid.
for ax in axes.ravel()[_n_layers:]:
    ax.set_visible(False)

fig.suptitle('Cross-fold selection stability vs panel size',
             fontsize=17)
fig.tight_layout()
plt.show()

## 8 · Parsimony & redundancy

In [ ]:
for metric, label in [('n_selected', 'Features selected'),
                      ('reduction_ratio', 'Reduction ratio'),
                      ('redundancy', 'Residual redundancy (mean |r|)')]:
    piv = summary.pivot(index='method', columns='layer', values=metric)
    print(f'\n{label}:')
    display(piv.style.background_gradient(
        cmap='viridis' if metric != 'redundancy' else 'viridis_r', axis=None).format('{:.3f}'))

## 9 · Statistical tests (per layer)

Friedman omnibus across methods, then pairwise Wilcoxon signed-rank vs.
SelectOmics (paired by fold, Holm–Bonferroni corrected) with Cliff's delta.
`favours='method'` means the challenger beat SelectOmics on that metric.
Tests are run **per layer** (not pooled), since the omic layers are biologically
distinct.

> **Read the p-values with these caveats** (a hostile reviewer will, so we state
> them up front):
> - Pairwise tests need ≥ 4 paired folds, use `FAST_MODE=False` (10 folds).
> - The 10 folds come from `RepeatedStratifiedKFold`, so their training sets
>   **overlap heavily** → folds are *not* independent blocks. Nominal p-values
>   are therefore **anti-conservative** (optimistic), and absolute Kuncheva
>   stability is **optimistic** for the same reason. They are valid for *ranking*
>   methods on identical folds, not as exact significance.
> - **SelectOmics carves its own internal validation split** (~20%) from each
>   training fold, so it selects on ~80% of the data every competitor selects on.
>   This is intrinsic to the method and *disadvantages* SelectOmics, it does not
>   inflate any SelectOmics win.
> - We do **not** standardise features (the layers are pre-normalised and z-scoring
>   would neutralise the variance-based steps). One consequence: the L1 penalties
>   in LASSO/ElasticNet are scale-sensitive, so those baselines run on native
>   scales, disclosed, and applied identically to the shared input.

In [ ]:
ALPHA = LGG_BENCHMARK_SPEC['alpha']
for layer in layers_present:
    sub = results[results['layer'] == layer]
    print(f'\n══ {layer} ══')
    for metric in ['auc_ovr', 'f1_macro']:
        fr = bl.friedman_test(sub, metric)
        sig = 'SIGNIFICANT' if (fr['p_value'] == fr['p_value'] and fr['p_value'] < ALPHA) else 'n.s.'
        print(f'  Friedman {metric:12s}: chi2={fr["statistic"]}  p={fr["p_value"]}  [{sig}]  (n_obs={fr["n_obs"]})')
    pw = bl.pairwise_vs_reference(sub, 'auc_ovr', reference='SelectOmics')
    if not pw.empty:
        display(pw[['method', 'n_pairs', 'p_corrected', 'significant',
                    'cliff_delta', 'effect_size', 'favours']])

## 10 · Biological soft-truth, do selected genes recover known glioma drivers?

Real data has no ground-truth informative set, but glioma is one of the
best-characterised cancers. We check whether each method's stable feature set
(selected in ≥ half the folds) recovers an established LGG driver panel. Most
meaningful for the gene-symbol layers (mRNA, CNV); methylation `cg` probes and
miRNAs need separate annotation, so a curated glioma-miRNA list is used for the
miRNA layer.

> **⚠ Why this metric is weak on these layers.** The subtype-defining events in
> glioma are *mutations* (IDH1/2, ATRX, TP53, CIC, FUBP1) and *broad* copy-number /
> methylation changes (1p/19q codeletion, G-CIMP), none of which appear as the
> named driver gene in an **expression** matrix (a point mutation barely changes
> mRNA level) or as a single gene in a per-gene **CNV** matrix (arm-scale signal is
> spread across hundreds of co-located genes). Empirically, driver-symbol recovery
> here mostly flags methods that keep *thousands* of features (they "recover"
> drivers by chance, enrichment ≈ 1×) and unfairly penalises parsimonious methods
> that legitimately select compact co-expressed proxies. **Do not read low driver
> recovery as "wrong features."** A valid biological check needs pathway/GO
> enrichment or a curated subtype-*expression* signature, not mutation-driver symbols.

In [ ]:
# Curated, well-established LGG / glioma markers (WHO 2021, TCGA 2015).
DRIVER_GENES = {
    'IDH1', 'IDH2', 'ATRX', 'TP53', 'CIC', 'FUBP1', 'TERT', 'EGFR', 'PTEN',
    'CDKN2A', 'CDKN2B', 'NF1', 'PIK3CA', 'PIK3R1', 'RB1', 'NOTCH1', 'PDGFRA',
    'CDK4', 'MDM2', 'MDM4', 'NRAS', 'BRAF', 'SMARCA4', 'ARID1A', 'MYCN', 'PTPN11',
}
DRIVER_MIRNAS = {  # recurrently glioma-associated miRNAs
    'hsa-mir-21', 'hsa-mir-10b', 'hsa-mir-181a', 'hsa-mir-181b', 'hsa-mir-221',
    'hsa-mir-222', 'hsa-mir-26a', 'hsa-mir-7', 'hsa-mir-128', 'hsa-mir-9',
    'hsa-mir-15b', 'hsa-mir-196a', 'hsa-mir-196b',
}

def _symbol(feature_name, layer):
    # Strip whichever modality suffix the feature carries (works for single-layer
    # and 'merged' runs alike). miRNA names are lowercased to match the panel.
    for suf in ('mRNA', 'Methy', 'miRNA', 'CNV'):
        if feature_name.endswith(f'_{suf}'):
            base = feature_name[:-(len(suf) + 1)]
            return base.lower() if suf == 'miRNA' else base.upper()
    return feature_name.upper()

def _panel(layer):
    if layer == 'miRNA':
        return {s.lower() for s in DRIVER_MIRNAS}
    if layer == 'Methy':
        return set()                       # cg probes need gene annotation, skipped
    if layer == 'merged':                  # mixed modalities → genes + miRNAs
        return DRIVER_GENES | {s.lower() for s in DRIVER_MIRNAS}
    return DRIVER_GENES

bio_rows = []
for layer in layers_present:
    _, _, feat_names = bl.get_layer(df, layer)
    panel = _panel(layer)
    n_fold = results[(results['layer'] == layer)]['fold'].nunique()
    for method, g in results[results['layer'] == layer].groupby('method'):
        from collections import Counter
        c = Counter()
        for s in g['selected_indices']:
            for i in str(s).split(','):
                if i != '':
                    c[int(i)] += 1
        stable = {idx for idx, n in c.items() if n >= max(1, n_fold // 2 + n_fold % 2)}
        syms = {_symbol(feat_names[i], layer) for i in stable}
        hits = sorted(syms & panel) if panel else []
        bio_rows.append({'layer': layer, 'method': method,
                         'stable_features': len(stable),
                         'driver_hits': len(hits),
                         'drivers': ', '.join(hits) if hits else (', (no gene annotation)' if not panel else '-')})

bio = pd.DataFrame(bio_rows)
print('Known-driver recovery among features stable across ≥half the folds:')
display(bio.sort_values(['layer', 'driver_hits'], ascending=[True, False]))

## 11 · Save results

In [ ]:
xlsx = OUTPUT_DIR / f'benchmark_lgg_{N_SPLITS * N_REPEATS}fold_full.xlsx'
with pd.ExcelWriter(xlsx) as writer:
    results.to_excel(writer, sheet_name='Raw', index=False)
    summary.to_excel(writer, sheet_name='Summary', index=False)
    bio.to_excel(writer, sheet_name='Biology', index=False)
print(f'Saved → {xlsx}')
print(f'Raw CSV → {RAW_CSV}')

#### 12 · Interpretation

Fill in once `FAST_MODE=False` has been run on all four layers. Expected reading,
consistent with the synthetic benchmark:

- **Where SelectOmics should lead:** the high-dimensional layers (mRNA, CNV,
  Methy ≈ 11k features each) on **parsimony** (fewest features, highest reduction)
  and **stability at low n_selected**, while holding **downstream AUC** on par
  with the field.
- **Where it should be merely competitive:** miRNA (only 328 features), not the
  small-n/large-p regime it targets, and any layer where a sparse linear model
  already separates the subtypes cleanly.
- **Biology:** stable mRNA/CNV selections should recover canonical glioma drivers
  (IDH1/2, ATRX, TP53, CIC, EGFR, CDKN2A …); this is the real-world payoff the
  synthetic benchmark cannot show.

### Scaling up

```python
FAST_MODE = False     # all four layers, 10 folds, SelectOmics Step 3 ON
```

This is a multi-hour run (the three ~11k-feature layers dominate). It is
checkpoint-safe, interrupt and re-run to resume. For a quick look at a single
heavy layer, set `LAYERS_TO_RUN = ['mRNA']` manually after the config cell.